In [1]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    model_name="qwen-plus",
    openai_api_key = os.getenv("DASHSCOPE_API_KEY"),
    openai_api_base = "https://dashscope.aliyuncs.com/compatible-mode/v1",
    temperature=0.9, 
    max_tokens=5000,
)

In [4]:
from langchain.agents import tool

@tool
def get_word_length(word: str)->int:
    "Return the length of the word"
    return len(word)

tools = [get_word_length]    

In [5]:
from langchain.schema import SystemMessage
from langchain.agents import OpenAIFunctionsAgent

system_message = SystemMessage(content = "你是非常强大的AI助手，但在计算单词长度方面不擅长。")
prompt = OpenAIFunctionsAgent.create_prompt(system_message=system_message)

agent = OpenAIFunctionsAgent(llm=llm, tools=tools, prompt=prompt)

/var/folders/70/5jwxzn313zb62r3ztdrxvytr0000gn/T/ipykernel_50291/1020565131.py:7: LangChainDeprecationWarning: The class `OpenAIFunctionsAgent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use :meth:`~create_openai_functions_agent` instead.
  agent = OpenAIFunctionsAgent(llm=llm, tools=tools, prompt=prompt)


In [6]:
from langchain.agents import AgentExecutor

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [7]:
agent_executor.invoke("单词“educa”中有多少个字母?")



> Entering new AgentExecutor chain...

Invoking: `get_word_length` with `{'word': 'educa'}`


5单词“educa”中有5个字母。

> Finished chain.


{'input': '单词“educa”中有多少个字母?', 'output': '单词“educa”中有5个字母。'}

In [9]:
from langchain.prompts import MessagesPlaceholder
MEMORY_KEY = "chat_history"
prompt=OpenAIFunctionsAgent.create_prompt(
    system_message=system_message,
    extra_prompt_messages=[MessagesPlaceholder(variable_name=MEMORY_KEY)]
)

In [14]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(memory_key=MEMORY_KEY, return_messages=True)

In [15]:
agent = OpenAIFunctionsAgent(llm=llm, tools=tools, prompt=prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, memory=memory, verbose=True)

In [16]:
agent_executor.invoke("单词“educa”中有多少个字母?")



> Entering new AgentExecutor chain...

Invoking: `get_word_length` with `{'word': 'educa'}`


5单词“educa”中有5个字母。

> Finished chain.


{'input': '单词“educa”中有多少个字母?',
 'chat_history': [HumanMessage(content='单词“educa”中有多少个字母?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='单词“educa”中有5个字母。', additional_kwargs={}, response_metadata={})],
 'output': '单词“educa”中有5个字母。'}